# Kaggle House Prices: improving my Random Forest

Competition: [Housing Prices Competition for Kaggle Learn Users](https://www.kaggle.com/c/home-data-for-ml-course) (Iowa houses).

My first version (Intro to ML course) was a Random Forest with 7 columns and a validation MAE of **21,857**. Now that I finished **Intermediate Machine Learning**, I want to bring that error down with: missing values, categorical columns, pipelines, cross-validation and XGBoost.

MAE = mean absolute error in dollars (lower is better).

## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# on Kaggle use: '../input/home-data-for-ml-course/train.csv' (and test.csv)
train_data = pd.read_csv('train.csv', index_col='Id')
test_data = pd.read_csv('test.csv', index_col='Id')

y = train_data.SalePrice
X = train_data.drop(['SalePrice'], axis=1)
test_X = test_data.copy()

print(X.shape, test_X.shape)

(1460, 79) (1459, 79)


## 2. Look at the columns and missing values

In [ ]:
# MSSubClass is a number but it is really a category (a code for the type of house)
X['MSSubClass'] = X['MSSubClass'].astype(str)
test_X['MSSubClass'] = test_X['MSSubClass'].astype(str)

num_cols = list(X.select_dtypes(include='number').columns)
cat_cols = list(X.select_dtypes(exclude='number').columns)
print(len(num_cols), 'numeric columns,', len(cat_cols), 'categorical columns')

missing = X.isnull().sum()
print()
print(missing[missing > 0].sort_values(ascending=False).head(10))

35 numeric columns, 44 categorical columns

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
dtype: int64


Many text columns are missing because the house simply doesn't have that thing (for example `PoolQC` is empty when there is no pool). So for text columns I fill the missing values with the word `"None"` instead of the most common value. Numeric columns get the median.

## 3. Old model (baseline)

I score every model with the same validation split as before and with 5-fold cross-validation.

In [3]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

old_features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']

model = RandomForestRegressor(random_state=1)
model.fit(train_X[old_features], train_y)
preds = model.predict(val_X[old_features])
print('Validation MAE:', round(mean_absolute_error(val_y, preds)))

scores = -1 * cross_val_score(model, X[old_features], y, cv=5, scoring='neg_mean_absolute_error')
print('CV MAE:', round(scores.mean()))

Validation MAE: 21857


CV MAE: 22918


## 4. All numeric columns + missing values

In [4]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(random_state=1))
])

num_pipe.fit(train_X[num_cols], train_y)
preds = num_pipe.predict(val_X[num_cols])
print('Validation MAE:', round(mean_absolute_error(val_y, preds)))

scores = -1 * cross_val_score(num_pipe, X[num_cols], y, cv=5, scoring='neg_mean_absolute_error')
print('CV MAE:', round(scores.mean()))

Validation MAE: 16712


CV MAE: 18039


## 5. Add the categorical columns

In [5]:
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', cat_transformer, cat_cols)
])

cat_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=1))
])

cat_pipe.fit(train_X, train_y)
preds = cat_pipe.predict(val_X)
print('Validation MAE:', round(mean_absolute_error(val_y, preds)))

scores = -1 * cross_val_score(cat_pipe, X, y, cv=5, scoring='neg_mean_absolute_error')
print('CV MAE:', round(scores.mean()))

Validation MAE: 16468


CV MAE: 17740


## 6. Add a few new columns

Some simple columns I made from the existing ones: total square feet, age of the house, age since remodel and total number of bathrooms.

In [6]:
def add_new_columns(df):
    df = df.copy()
    df['TotalSF'] = df['TotalBsmtSF'].fillna(0) + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath']
                       + df['BsmtFullBath'].fillna(0) + 0.5 * df['BsmtHalfBath'].fillna(0))
    return df

X2 = add_new_columns(X)
test_X2 = add_new_columns(test_X)
num_cols2 = list(X2.select_dtypes(include='number').columns)

preprocessor2 = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols2),
    ('cat', cat_transformer, cat_cols)
])

rf_pipe2 = Pipeline([
    ('preprocessor', preprocessor2),
    ('model', RandomForestRegressor(random_state=1))
])

train_X2, val_X2, train_y, val_y = train_test_split(X2, y, random_state=1)
rf_pipe2.fit(train_X2, train_y)
preds = rf_pipe2.predict(val_X2)
print('Validation MAE:', round(mean_absolute_error(val_y, preds)))

scores = -1 * cross_val_score(rf_pipe2, X2, y, cv=5, scoring='neg_mean_absolute_error')
print('CV MAE:', round(scores.mean()))

Validation MAE: 16334


CV MAE: 17231


## 7. XGBoost

I train XGBoost on `log(SalePrice)`, because house prices are skewed and this usually helps. The predictions are converted back to dollars, so the MAE below is still in dollars.

For each `max_depth` I use early stopping to find how many trees to use, then I check it with cross-validation.

In [7]:
train_X_ready = preprocessor2.fit_transform(train_X2)
val_X_ready = preprocessor2.transform(val_X2)

best_setups = []
for depth in [2, 3, 4]:
    xgb = XGBRegressor(n_estimators=3000, learning_rate=0.03, max_depth=depth,
                       subsample=0.8, colsample_bytree=0.5,
                       early_stopping_rounds=50, eval_metric='mae', random_state=1)
    xgb.fit(train_X_ready, np.log(train_y), eval_set=[(val_X_ready, np.log(val_y))], verbose=False)
    n_trees = xgb.best_iteration + 1

    # score in dollars
    val_preds = np.exp(xgb.predict(val_X_ready))
    val_mae = mean_absolute_error(val_y, val_preds)

    # cross-validation with this number of trees
    pipe = Pipeline([
        ('preprocessor', preprocessor2),
        ('model', TransformedTargetRegressor(
            regressor=XGBRegressor(n_estimators=n_trees, learning_rate=0.03, max_depth=depth,
                                   subsample=0.8, colsample_bytree=0.5, random_state=1),
            func=np.log, inverse_func=np.exp))
    ])
    cv_mae = -1 * cross_val_score(pipe, X2, y, cv=5, scoring='neg_mean_absolute_error').mean()

    print('max_depth', depth, '| trees', n_trees, '| validation MAE', round(val_mae), '| CV MAE', round(cv_mae))
    best_setups.append((cv_mae, depth, n_trees))

max_depth 2 | trees 674 | validation MAE 13558 | CV MAE 14906


max_depth 3 | trees 558 | validation MAE 13691 | CV MAE 14548


max_depth 4 | trees 511 | validation MAE 14103 | CV MAE 14640


## 8. Final model and submission

I take the setup with the lowest CV MAE, train it on all the training data and predict the test set.

In [8]:
best_cv, best_depth, best_trees = min(best_setups)
print('best:', 'max_depth', best_depth, 'trees', best_trees, 'CV MAE', round(best_cv))

final_model = Pipeline([
    ('preprocessor', preprocessor2),
    ('model', TransformedTargetRegressor(
        regressor=XGBRegressor(n_estimators=best_trees, learning_rate=0.03, max_depth=best_depth,
                               subsample=0.8, colsample_bytree=0.5, random_state=1),
        func=np.log, inverse_func=np.exp))
])
final_model.fit(X2, y)

test_preds = final_model.predict(test_X2)

output = pd.DataFrame({'Id': test_X2.index, 'SalePrice': test_preds})
output.to_csv('submission.csv', index=False)
print(output.head())

best: max_depth 3 trees 558 CV MAE 14548


     Id      SalePrice
0  1461  123853.093750
1  1462  158216.375000
2  1463  181643.156250
3  1464  190076.671875
4  1465  193160.750000


## What I learned

- Missing values and text columns can be handled with `SimpleImputer` and `OneHotEncoder`, so I can use all columns, not only 7.
- In this data a missing text value often means "this house doesn't have it", so filling with `"None"` makes sense.
- Pipelines keep preprocessing and model together and avoid leaking validation data into training.
- Cross-validation gives a more trustworthy score than one validation split.
- XGBoost with early stopping, plus a log of the price, was the best model I tried.

## Ideas for later

- Treat quality columns (`ExterQual`, `KitchenQual`, ...) as ordered levels
- Try more values for `learning_rate`, `min_child_weight`, regularization
- Mix (blend) XGBoost with other models